# Experiment 14 — Mamba (Selective State Space Model)

New track. Separate from the biological-neuron / Hebbian conversational-agent line
(experiments 01, 08-12): this notebook and the next three (15 RWKV, 16 Gated DeltaNet,
17 Kimi Linear) test modern **subquadratic alternatives to standard Transformer
self-attention** — the same kind of attention built from scratch in the main
curriculum's `module_10_scaled_dot_product_attention` through `module_11_multi_head_attention`.

Standard attention costs **O(T²)** compute and memory in sequence length `T`, because
every token looks back at every earlier token directly (that's the whole point — exact,
lossless lookback). These four architectures trade that away: each one compresses
everything seen so far into a **fixed-size recurrent state**, updated with O(1) work per
new token — O(T) total, and a constant-size cache regardless of how long the sequence
gets. That's a real, unresolved research question, not a free lunch: can a fixed-size
state actually hold onto what a full O(T) attention cache holds onto?

**Mamba** (Gu & Dao, 2023) is the architecture that reignited interest in this question.
It descends from S4 (structured state space sequence models) — a continuous-time linear
system `h'(t) = Ah(t) + Bx(t)`, discretized into a recurrence `h_t = Ā h_{t-1} + B̄ x_t`.
Mamba's one change, the **"S6" / selective SSM**, is making the discretization step size
`Δ` (and `B`, `C`) *functions of the current input token* instead of fixed constants — so
the model can decide, per token, whether to hang onto its state (small `Δ`, slow decay) or
overwrite it (large `Δ`). This notebook builds that mechanism from scratch (no
`mamba-ssm` package) and tests it on a task built to stress exactly the fixed-state
question above.

## The test: multi-query associative recall (MQAR)

All four notebooks in this track (14-17) use the same task, so results are directly
comparable across them. It's the standard benchmark this literature actually uses (Arora
et al. 2023, "Zoology: Measuring and Improving Recall in Efficient Language Models";
reused by the Gated DeltaNet and Kimi Linear papers built in 16/17) precisely because it
isolates the fixed-state question:

- Show the model `K` key→value pairs, presented in a random order. The key→value
  binding is **freshly randomized every single example** — nothing can be memorized into
  the weights, every example is a genuine in-context lookup.
- Then show `M` query keys (drawn with repetition from the same `K` keys) and ask the
  model to output the value bound to each one.
- Standard full self-attention solves this **trivially**: a query token can attend
  directly back to its matching key token, wherever it is. The open question is whether a
  fixed-size state, built one token at a time and never revisited, can hold the same
  information.
- Chance-level accuracy is `1/K` (guessing one of the `K` possible values).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Vocabulary layout: 0=unused, 1=SEP, keys = 2..2+K-1, values = 2+K..2+2K-1
K = 8   # number of key-value pairs per example (= number of possible keys/values)
M = 4   # number of queries per example
SEP = 1
KEY0 = 2
VAL0 = 2 + K
VOCAB = 2 + 2 * K


def make_batch(batch_size, device="cpu"):
    # random bijection: key-identity i (0..K-1) is bound to a randomly assigned value class
    value_assignment = torch.argsort(torch.rand(batch_size, K), dim=1)
    # random order the K pairs are presented in
    order = torch.argsort(torch.rand(batch_size, K), dim=1)

    key_ids = KEY0 + order
    val_ids = VAL0 + torch.gather(value_assignment, 1, order)

    context = torch.stack([key_ids, val_ids], dim=2).reshape(batch_size, 2 * K)
    sep = torch.full((batch_size, 1), SEP, dtype=torch.long)

    q_key_identity = torch.randint(0, K, (batch_size, M))
    query_tokens = KEY0 + q_key_identity
    query_labels = torch.gather(value_assignment, 1, q_key_identity)

    seq = torch.cat([context, sep, query_tokens], dim=1)
    return seq.to(device), query_labels.to(device)


seq, labels = make_batch(1)
print("one example sequence:", seq[0].tolist())
print("(context pairs, then SEP=1, then", M, "query keys)")
print("correct value-class for each query:", labels[0].tolist())


device: cuda
one example sequence: [6, 10, 7, 15, 8, 11, 4, 14, 5, 16, 2, 12, 9, 17, 3, 13, 1, 2, 6, 4, 3]
(context pairs, then SEP=1, then 4 query keys)
correct value-class for each query: [2, 0, 4, 3]


## The selective SSM (S6) mechanism

For each channel, the continuous-time system `h'(t) = Ah(t) + Bx(t)`, `y(t) = Ch(t)` is
discretized (zero-order hold) into:

```
Ā_t = exp(Δ_t · A)
h_t   = Ā_t · h_{t-1} + Δ_t · B_t · x_t
y_t   = C_t · h_t
```

In plain S4, `A`, `B`, `C`, `Δ` are all fixed (learned once, same for every token). Mamba's
selectivity makes `Δ`, `B`, and `C` **linear projections of the current input** `x_t`
(with `Δ` passed through `softplus` to keep it positive) — only `A` stays fixed per
channel, parameterized as `-exp(A_log)` so it's always negative (a stable decay). This
lets the model choose, token by token, how much to trust old state vs. the new input.

The full Mamba block wraps this scan with the same pieces as the paper's diagram: an
input projection splitting into the SSM path `x` and a gate `z`, a short causal depthwise
conv1d on `x` (lets nearby tokens mix a little before the scan — turns out to be load-bearing,
see the results below), a SiLU activation, the selective scan itself, a skip connection
(`D`), and a final `SiLU(z)` gate before the output projection.

One thing this notebook deliberately does **not** reproduce: the real Mamba implementation
computes the scan with a hardware-aware parallel-scan CUDA kernel for speed. Here it's a
plain sequential Python loop over time steps — same math, much slower, but this is a
correctness/behavior test on a 21-token toy sequence, not a throughput benchmark.

In [2]:
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, conv_k=3):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.conv_k = conv_k
        self.in_proj = nn.Linear(d_model, 2 * d_model)          # -> (x, gate z)
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model, bias=True)
        self.x_proj = nn.Linear(d_model, 2 * d_state)            # -> (B_t, C_t), input-dependent
        self.dt_proj = nn.Linear(d_model, d_model)               # -> delta_t, input-dependent
        A_init = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_model, 1)
        self.A_log = nn.Parameter(torch.log(A_init))             # A = -exp(A_log), fixed per channel
        self.D = nn.Parameter(torch.ones(d_model))               # skip connection
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, Dm = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        # causal depthwise conv: pad only on the left so position t never sees t+1..
        x_pad = F.pad(x_in.transpose(1, 2), (self.conv_k - 1, 0))
        x_conv = F.silu(self.conv(x_pad).transpose(1, 2))

        delta = F.softplus(self.dt_proj(x_conv))          # (B,T,d_model), the "selectivity"
        B_t, C_t = self.x_proj(x_conv).chunk(2, dim=-1)    # (B,T,d_state) each, also input-dependent
        A = -torch.exp(self.A_log)                          # (d_model, d_state), fixed

        h = x.new_zeros(B, Dm, self.d_state)
        ys = []
        for t in range(T):
            dt = delta[:, t, :]
            dA = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0))                              # (B,d_model,d_state)
            dBx = dt.unsqueeze(-1) * B_t[:, t, :].unsqueeze(1) * x_conv[:, t, :].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_t[:, t, :].unsqueeze(1)).sum(-1)
            ys.append(y_t)
        y = torch.stack(ys, dim=1) + x_conv * self.D
        y = y * F.silu(z)
        return self.out_proj(y)


## Wiring it into a tiny sequence model

Two stacked `MambaBlock`s (pre-norm residual, like a Transformer block) sit on top of a
token embedding. A linear classifier head reads the hidden state at every position; loss
is cross-entropy computed **only** at the `M` query positions (each query position's
hidden state must encode enough about the matching earlier key to name its value —
that's the entire test). Training uses freshly sampled random key/value bindings every
step (infinite data), so nothing can be solved by memorizing fixed pairs.

In [3]:
class TokenModel(nn.Module):
    def __init__(self, layer_factory, d_model, n_layers, vocab=VOCAB, n_classes=K):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.layers = nn.ModuleList([layer_factory() for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, tokens):
        x = self.embed(tokens)
        for layer, norm in zip(self.layers, self.norms):
            x = x + layer(norm(x))
        return self.head(self.final_norm(x))


def train_and_eval(model, steps=1500, batch_size=64, lr=3e-3, n_queries=M, log_every=300):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for step in range(steps):
        seq, labels = make_batch(batch_size, device)
        logits = model(seq)[:, -n_queries:, :]
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            acc = (logits.argmax(-1) == labels).float().mean().item()
            history.append((step, loss.item(), acc))
            print(f"step {step:4d}  loss {loss.item():.4f}  train_acc {acc:.3f}")

    model.eval()
    with torch.no_grad():
        seq, labels = make_batch(2000, device)
        logits = model(seq)[:, -n_queries:, :]
        test_acc = (logits.argmax(-1) == labels).float().mean().item()
    print(f"\nFINAL TEST ACC: {test_acc:.4f}   (chance = {1/K:.4f})")
    return history, test_acc


torch.manual_seed(0)
mamba_model = TokenModel(lambda: MambaBlock(64, d_state=16), d_model=64, n_layers=2)
n_params = sum(p.numel() for p in mamba_model.parameters())
print(f"params: {n_params:,}")
history, test_acc = train_and_eval(mamba_model)


params: 42,184


step    0  loss 2.2023  train_acc 0.137


step  300  loss 2.0819  train_acc 0.117


step  600  loss 2.0844  train_acc 0.148


step  900  loss 0.4539  train_acc 0.816


step 1200  loss 0.0007  train_acc 1.000


step 1499  loss 0.0003  train_acc 1.000

FINAL TEST ACC: 1.0000   (chance = 0.1250)


## What actually happened

**The selective SSM solved the task, but not smoothly — it sat at chance level for
roughly half of training, then snapped to perfect accuracy in a short window.** Train
accuracy was flat around 0.12-0.15 (chance = 0.125) through step 600, jumped to 0.816 by
step 900, then hit 1.000 by step 1200 and stayed there. Final held-out test accuracy:
**1.0000** on 2,000 fresh examples (chance = 0.1250), with 42,184 parameters.

This sudden-transition shape (long plateau, then a fast climb) is a real and informative
result, not noise — it looks like the model spending most of training searching for the
right *use* of its selectivity mechanism (learning to set delta large at key/value
positions so they get written into state, and shaping B and C so a query's projection
pulls out the right stored value) rather than gradually improving a soft heuristic. Worth
watching for in the next three notebooks: does every one of these architectures show this
same late phase-transition shape, or is it specific to Mamba's particular parameterization?

**A real bug caught during development, worth remembering:** the first version of
`MambaBlock` had no causal conv before the scan, so the equivalent of `k_t`/`v_t` was
computed from the *current* token alone. That's fatal for this task specifically —
writing "key to value" into the state requires seeing the key together with its value at
the moment of the write, and a single token in isolation never sees both. Every
architecture in this track (14-17) needs some form of short local mixing (Mamba's causal
`conv1d`, RWKV's token-shift, the delta-rule notebooks' short conv on q/k/v) purely so a
token can "see" the one before it before the recurrent state update happens. Without it,
every model in this series is stuck at exact chance no matter how long it trains —
confirmed by re-running without the fix first.

**What this does and doesn't show:** this is a single seed, toy scale (`K=8` pairs,
`T=21` tokens, `d_state=16`), and a sequential Python loop rather than the real
hardware-aware parallel scan — a correctness/behavior demonstration, not a throughput or
capacity benchmark. Whether a *fixed* 16-dimensional state per channel keeps working as
`K` grows well past `d_state` is exactly the kind of question the delta-rule architectures
in 16 and 17 are designed to address more robustly — not tested here.